# CodexProbe on Kaggle
Records every LLM call the Codex CLI makes, then swaps Codex's backend from OpenAI to a local Qwen coder model by changing only the config.

Settings: **GPU T4 x2**, **Internet on**. Optional: Add-ons → Secrets → `OPENAI_API_KEY` for run (a); without it run (a) is skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install CodexProbe and the Codex CLI

In [ ]:
import os, shutil, subprocess, time, urllib.request, json
WORK = "/kaggle/working/codex-probe"   # not named codex_probe, so it can't shadow the package
if not os.path.isdir(WORK):
    !git clone -q https://github.com/Rahul-Verma97/codex-probe.git {WORK}
!pip install -q {WORK}
if shutil.which("npm") is None:
    !curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null && apt-get install -y -qq nodejs > /dev/null
!npm i -g @openai/codex > /dev/null 2>&1
!codex --version

## 2. Serve Qwen locally with Ollama
Codex only accepts `wire_api = "responses"` now, so the server must expose `/v1/responses` (Ollama does).
Codex's first request is ~10k tokens, but Ollama's default context is 4k and it truncates silently, so each model gets a copy with `num_ctx 32768` baked in.

In [ ]:
!apt-get update -qq > /dev/null && apt-get install -y -qq zstd > /dev/null   # Ollama releases are .tar.zst
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
assert shutil.which("ollama"), "Ollama install failed"
subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2); break
    except Exception:
        time.sleep(1)
MODELS = {"qwen2.5-coder:7b": "qwen2.5-coder:7b-32k", "qwen3-coder:30b": "qwen3-coder:30b-32k"}
for base, tagged in MODELS.items():
    !ollama pull {base} > /dev/null
    open("/tmp/Modelfile", "w").write(f"FROM {base}\nPARAMETER num_ctx 32768\n")
    !ollama create {tagged} -f /tmp/Modelfile > /dev/null
!ollama list

In [ ]:
# Does each model return a real function_call through /v1/responses (not JSON written as text)?
tool = {"type": "function", "name": "exec_command", "description": "Run a shell command",
        "parameters": {"type": "object", "properties": {"cmd": {"type": "string"}}, "required": ["cmd"]}}
for m in MODELS.values():
    body = json.dumps({"model": m, "input": "List the files in the current directory.", "tools": [tool]}).encode()
    req = urllib.request.Request("http://127.0.0.1:11434/v1/responses", body, {"Content-Type": "application/json"})
    out = json.load(urllib.request.urlopen(req, timeout=600))["output"]
    print(m, "->", [(o["type"], o.get("name"), o.get("arguments") or o.get("content")) for o in out])

## 3. Run (a): Codex → CodexProbe → OpenAI (needs the secret)

In [ ]:
%cd {WORK}
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["OPENAI_API_KEY"] = UserSecretsClient().get_secret("OPENAI_API_KEY")
except Exception:
    print("No OPENAI_API_KEY secret attached; skipping run (a).")
if os.environ.get("OPENAI_API_KEY"):
    !python examples/quickstart.py --config configs/openai.json --model gpt-5-codex --sandbox danger-full-access

## 4. Run (b): same recorder, backend swapped by config → local Qwen

In [ ]:
for m in MODELS.values():
    print(f"\n========== {m} ==========")
    !python examples/quickstart.py --config configs/qwen_ollama.json --model {m} --env-key OLLAMA_API_KEY --sandbox danger-full-access --timeout 1800

## 5. Inspect the logs

In [ ]:
import glob
from codex_probe import load_session
for path in sorted(glob.glob("codex_probe_logs/*.jsonl")):
    calls = load_session(path)                       # schema-validates every line
    print(f"\n== {path}")
    !codex-probe show {path}
    if calls:                                        # what Codex actually sent in the first call
        req = calls[0]["request_body"]
        print("request keys:", list(req))
        print("tools:", [t.get("name") or t.get("type") for t in req.get("tools", [])])
        print("input roles:", [i.get("role") or i.get("type") for i in req.get("input", [])])
        print("served by:", (calls[0].get("response_body") or {}).get("model"), "| upstream:", calls[0]["upstream_url"])
!mkdir -p /kaggle/working/results && cp -r codex_probe_logs /kaggle/working/ollama.log /kaggle/working/results/